# Colab-ready 教學入口：Ch15 使用 CNN 進行影像分類

本 notebook 由官方程式碼 notebook 產生，第一格加入 Colab setup，讓學生不需要手動 clone repo 或切換工作目錄。

- 官方來源：`ch15/ch15_part1.ipynb`
- 官方 repo：https://github.com/rasbt/python-machine-learning-book-3rd-edition.git
- 書籍：Sebastian Raschka and Vahid Mirjalili, *Python Machine Learning, 3rd Ed.*, Packt Publishing, 2019
- 程式碼授權：MIT License，請參考本 repo 的 `THIRD_PARTY_NOTICES.md`

上課時請先執行下一格 setup，再依序執行原 notebook。若深度學習或大型資料章節耗時過久，請改用課堂 quick mode 或由講師示範重點 cell。


In [ ]:
# @title Colab setup for Python Machine Learning 3rd ed.
import importlib
import os
import platform
import subprocess
import sys

REPO_URL = "https://github.com/rasbt/python-machine-learning-book-3rd-edition.git"
REPO_DIR = "/content/python-machine-learning-book-3rd-edition" if os.path.exists("/content") else os.path.abspath("_python_ml_3e_official")
CHAPTER_DIR = "ch15"
EXTRA_PACKAGES = ['watermark', 'mlxtend', 'pyprind', 'tensorflow', 'tensorflow-datasets']

def _run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])

target_dir = os.path.join(REPO_DIR, CHAPTER_DIR)
os.chdir(target_dir)
print("Working directory:", os.getcwd())

def ensure_import(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        _run([sys.executable, "-m", "pip", "install", "-q", pip_name])

for import_name, pip_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("sklearn", "scikit-learn"),
    ("scipy", "scipy"),
]:
    ensure_import(import_name, pip_name)

for pkg in EXTRA_PACKAGES:
    import_name = pkg.split("==")[0].split("[")[0].replace("-", "_")
    if pkg.startswith("tensorflow-datasets"):
        import_name = "tensorflow_datasets"
    if pkg.startswith("scikit-learn"):
        import_name = "sklearn"
    if pkg.startswith("gym=="):
        import_name = "gym"
    ensure_import(import_name, pkg)

import numpy as np

# Compatibility shims for the current Colab runtime family.
# Colab 2026.04 lists Python 3.12.13, NumPy 2.0.2, and TensorFlow 2.19.0.
# The 2019 book notebooks still use a few aliases/API shapes from older releases.
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

try:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.rcParams["figure.figsize"] = (7, 5)
except Exception as exc:
    print("matplotlib setup skipped:", type(exc).__name__, exc)

try:
    import tensorflow as tf
    print("TensorFlow devices:", [device.device_type + ":" + device.name.split(":")[-1] for device in tf.config.list_physical_devices()])
except Exception as exc:
    print("TensorFlow not loaded:", type(exc).__name__, exc)

try:
    import gym

    if not getattr(gym, "_pyml3e_old_api_patch", False):
        _original_gym_make = gym.make

        class _OldStepAPIWrapper(gym.Wrapper):
            def reset(self, *args, **kwargs):
                result = self.env.reset(*args, **kwargs)
                if isinstance(result, tuple) and len(result) == 2:
                    return result[0]
                return result

            def step(self, action):
                result = self.env.step(action)
                if isinstance(result, tuple) and len(result) == 5:
                    obs, reward, terminated, truncated, info = result
                    return obs, reward, bool(terminated or truncated), info
                return result

        def _patched_make(*args, **kwargs):
            env = _original_gym_make(*args, **kwargs)
            return _OldStepAPIWrapper(env)

        gym.make = _patched_make
        gym._pyml3e_old_api_patch = True
        print("Gym old-step API wrapper enabled.")
except Exception as exc:
    print("Gym compatibility setup skipped:", type(exc).__name__, exc)

print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
for mod_name in ["numpy", "pandas", "matplotlib", "sklearn", "tensorflow", "tensorflow_datasets", "gym"]:
    try:
        mod = importlib.import_module(mod_name)
        print(f"{mod_name}:", getattr(mod, "__version__", "installed"))
    except Exception as exc:
        print(f"{mod_name}: not loaded ({type(exc).__name__})")

print("Setup complete. Run the notebook cells below in order.")


*Python Machine Learning 3rd Edition* by [Sebastian Raschka](https://sebastianraschka.com) & [Vahid Mirjalili](http://vahidmirjalili.com), Packt Publishing Ltd. 2019

Code Repository: https://github.com/rasbt/python-machine-learning-book-3rd-edition

Code License: [MIT License](https://github.com/rasbt/python-machine-learning-book-3rd-edition/blob/master/LICENSE.txt)

# Chapter 15: Classifying Images with Deep Convolutional Neural Networks (Part 1/2)

Note that the optional watermark extension is a small IPython notebook plugin that I developed to make the code reproducible. You can just skip the following line(s).

In [ ]:
%load_ext watermark
%watermark -a "Sebastian Raschka & Vahid Mirjalili" -u -d -p numpy,scipy,matplotlib,tensorflow,tensorflow_datasets


In [ ]:
from IPython.display import Image
%matplotlib inline


##  The building blocks of convolutional neural networks

### Understanding CNNs and feature hierarchies

In [ ]:
Image(filename='images/15_01.png', width=700)




### Performing discrete convolutions

### Discrete convolutions in one dimension



In [ ]:
Image(filename='images/15_02.png', width=700)


In [ ]:
Image(filename='images/15_03.png', width=700)


### Padding inputs to control the size of the output feature maps



In [ ]:
Image(filename='images/15_04.png', width=700)


### Determining the size of the convolution output

In [ ]:
import tensorflow as tf
import numpy as np

print('TensorFlow version:', tf.__version__)
print('NumPy version: ', np.__version__)


In [ ]:
def conv1d(x, w, p=0, s=1):
    w_rot = np.array(w[::-1])
    x_padded = np.array(x)
    if p > 0:
        zero_pad = np.zeros(shape=p)
        x_padded = np.concatenate(
            [zero_pad, x_padded, zero_pad])
    res = []
    for i in range(0, int((len(x_padded) - len(w_rot)) / s) + 1, s):
        res.append(np.sum(
            x_padded[i:i+w_rot.shape[0]] * w_rot))
    return np.array(res)


## Testing:
x = [1, 3, 2, 4, 5, 6, 1, 3]
w = [1, 0, 3, 1, 2]

print('Conv1d Implementation:',
      conv1d(x, w, p=2, s=1))

print('Numpy Results:',
      np.convolve(x, w, mode='same')) 


### Performing a discrete convolution in 2D

In [ ]:
Image(filename='images/15_05.png', width=700)


In [ ]:
Image(filename='images/15_06.png', width=600)


In [ ]:
Image(filename='images/15_07.png', width=800)


In [ ]:
import scipy.signal


def conv2d(X, W, p=(0, 0), s=(1, 1)):
    W_rot = np.array(W)[::-1,::-1]
    X_orig = np.array(X)
    n1 = X_orig.shape[0] + 2*p[0]
    n2 = X_orig.shape[1] + 2*p[1]
    X_padded = np.zeros(shape=(n1, n2))
    X_padded[p[0]:p[0]+X_orig.shape[0],
    p[1]:p[1]+X_orig.shape[1]] = X_orig

    res = []
    for i in range(0, int((X_padded.shape[0] - 
                           W_rot.shape[0])/s[0])+1, s[0]):
        res.append([])
        for j in range(0, int((X_padded.shape[1] - 
                               W_rot.shape[1])/s[1])+1, s[1]):
            X_sub = X_padded[i:i+W_rot.shape[0],
                             j:j+W_rot.shape[1]]
            res[-1].append(np.sum(X_sub * W_rot))
    return(np.array(res))

X = [[1, 3, 2, 4], [5, 6, 1, 3], [1, 2, 0, 2], [3, 4, 3, 2]]
W = [[1, 0, 3], [1, 2, 1], [0, 1, 1]]

print('Conv2d Implementation:\n',
    conv2d(X, W, p=(1, 1), s=(1, 1)))


print('SciPy Results:\n',
    scipy.signal.convolve2d(X, W, mode='same'))


## Subsampling layers

In [ ]:
Image(filename='images/15_08.png', width=700)


## Putting everything together – implementing a CNN

### Working with multiple input or color channels



In [ ]:
Image(filename='images/15_09.png', width=800)


**TIP: Reading an image file**

In [ ]:
import tensorflow as tf


img_raw = tf.io.read_file('example-image.png')
img = tf.image.decode_image(img_raw)
print('Image shape:', img.shape)
print('Number of channels:', img.shape[2])
print('Image data type:', img.dtype)
print(img[100:102, 100:102, :])


In [ ]:
import imageio


img = imageio.imread('example-image.png')
print('Image shape:', img.shape)
print('Number of channels:', img.shape[2])
print('Image data type:', img.dtype)
print(img[100:102, 100:102, :])


**INFO-BOX: The rank of a grayscale image for input to a CNN**

In [ ]:
img_raw = tf.io.read_file('example-image-gray.png')
img = tf.image.decode_image(img_raw)
tf.print('Rank:', tf.rank(img))
tf.print('Shape:', img.shape)


In [ ]:
img = imageio.imread('example-image-gray.png')
tf.print('Rank:', tf.rank(img))
tf.print('Shape:', img.shape)

img_reshaped = tf.reshape(img, (img.shape[0], img.shape[1], 1))
tf.print('New Shape:', img_reshaped.shape)


## Regularizing a neural network with dropout



In [ ]:
Image(filename='images/15_10.png', width=700)


In [ ]:
from tensorflow import keras


conv_layer = keras.layers.Conv2D(
    filters=16, kernel_size=(3, 3),
    kernel_regularizer=keras.regularizers.l2(0.001))

fc_layer = keras.layers.Dense(
    units=16, kernel_regularizer=keras.regularizers.l2(0.001))


## Loss Functions for Classification

 * **`BinaryCrossentropy()`**
   * `from_logits=False` 
   * `from_logits=True`

 * **`CategoricalCrossentropy()`**
   * `from_logits=False`
   * `from_logits=True`
   
 * **`SparseCategoricalCrossentropy()`**
   * `from_logits=False`
   * `from_logits=True`


In [ ]:
Image(filename='images/15_11.png', width=800)


In [ ]:
from distutils.version import LooseVersion as Version


####### Binary Crossentropy
bce_probas = tf.keras.losses.BinaryCrossentropy(from_logits=False)
bce_logits = tf.keras.losses.BinaryCrossentropy(from_logits=True)

logits = tf.constant([0.8])
probas = tf.keras.activations.sigmoid(logits)

if Version(tf.__version__) >= '2.3.0':
    tf.print(
        'CCE (w Probas): {:.4f}'.format(
        cce_probas(y_true=[[0, 0, 1]], y_pred=probas)),
        '(w Logits): {:.4f}'.format(
        cce_logits(y_true=[[0, 0, 1]], y_pred=logits)))
    
else:
    tf.print(
        'CCE (w Probas): {:.4f}'.format(
        cce_probas(y_true=[0, 0, 1], y_pred=probas)),
        '(w Logits): {:.4f}'.format(
        cce_logits(y_true=[0, 0, 1], y_pred=logits)))


####### Categorical Crossentropy
cce_probas = tf.keras.losses.CategoricalCrossentropy(
    from_logits=False)
cce_logits = tf.keras.losses.CategoricalCrossentropy(
    from_logits=True)

logits = tf.constant([[1.5, 0.8, 2.1]])
probas = tf.keras.activations.softmax(logits)

tf.print(
    'CCE (w Probas): {:.4f}'.format(
    cce_probas(y_true=[0, 0, 1], y_pred=probas)),
    '(w Logits): {:.4f}'.format(
    cce_logits(y_true=[0, 0, 1], y_pred=logits)))

####### Sparse Categorical Crossentropy
sp_cce_probas = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=False)
sp_cce_logits = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True)

tf.print(
    'Sparse CCE (w Probas): {:.4f}'.format(
    sp_cce_probas(y_true=[2], y_pred=probas)),
    '(w Logits): {:.4f}'.format(
    sp_cce_logits(y_true=[2], y_pred=logits)))


## Implementing a deep convolutional neural network using TensorFlow

### The multilayer CNN architecture

In [ ]:
Image(filename='images/15_12.png', width=800)


### Loading and preprocessing the data

In [ ]:
import tensorflow_datasets as tfds
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
## MNIST dataset

mnist_bldr = tfds.builder('mnist')
mnist_bldr.download_and_prepare()
datasets = mnist_bldr.as_dataset(shuffle_files=False)
print(datasets.keys())
mnist_train_orig, mnist_test_orig = datasets['train'], datasets['test']


In [ ]:
BUFFER_SIZE = 10000
BATCH_SIZE = 64
NUM_EPOCHS = 20


In [ ]:
mnist_train = mnist_train_orig.map(
    lambda item: (tf.cast(item['image'], tf.float32)/255.0, 
                  tf.cast(item['label'], tf.int32)))

mnist_test = mnist_test_orig.map(
    lambda item: (tf.cast(item['image'], tf.float32)/255.0, 
                  tf.cast(item['label'], tf.int32)))

tf.random.set_seed(1)

mnist_train = mnist_train.shuffle(buffer_size=BUFFER_SIZE,
                                  reshuffle_each_iteration=False)

mnist_valid = mnist_train.take(10000).batch(BATCH_SIZE)
mnist_train = mnist_train.skip(10000).batch(BATCH_SIZE)


### Implementing a CNN using the TensorFlow Keras API

#### Configuring CNN layers in Keras

 * **Conv2D:** `tf.keras.layers.Conv2D`
   * `filters`
   * `kernel_size`
   * `strides`
   * `padding`
   
   
 * **MaxPool2D:** `tf.keras.layers.MaxPool2D`
   * `pool_size`
   * `strides`
   * `padding`
   
   
 * **Dropout** `tf.keras.layers.Dropout2D`
   * `rate`

### Constructing a CNN in Keras

In [ ]:
model = tf.keras.Sequential()

model.add(tf.keras.layers.Conv2D(
    filters=32, kernel_size=(5, 5),
    strides=(1, 1), padding='same',
    data_format='channels_last',
    name='conv_1', activation='relu'))

model.add(tf.keras.layers.MaxPool2D(
    pool_size=(2, 2), name='pool_1'))
    
model.add(tf.keras.layers.Conv2D(
    filters=64, kernel_size=(5, 5),
    strides=(1, 1), padding='same',
    name='conv_2', activation='relu'))

model.add(tf.keras.layers.MaxPool2D(pool_size=(2, 2), name='pool_2'))


In [ ]:
model.compute_output_shape(input_shape=(16, 28, 28, 1))


In [ ]:
    model.add(tf.keras.layers.Flatten())
    
model.compute_output_shape(input_shape=(16, 28, 28, 1))


In [ ]:
model.add(tf.keras.layers.Dense(
    units=1024, name='fc_1', 
    activation='relu'))

model.add(tf.keras.layers.Dropout(
    rate=0.5))
    
model.add(tf.keras.layers.Dense(
    units=10, name='fc_2',
    activation='softmax'))


In [ ]:
tf.random.set_seed(1)
model.build(input_shape=(None, 28, 28, 1))

model.compute_output_shape(input_shape=(16, 28, 28, 1))


In [ ]:
model.summary()


In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy']) # same as `tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')`

history = model.fit(mnist_train, epochs=NUM_EPOCHS, 
                    validation_data=mnist_valid, 
                    shuffle=True)


In [ ]:
hist = history.history
x_arr = np.arange(len(hist['loss'])) + 1

fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(1, 2, 1)
ax.plot(x_arr, hist['loss'], '-o', label='Train loss')
ax.plot(x_arr, hist['val_loss'], '--<', label='Validation loss')
ax.set_xlabel('Epoch', size=15)
ax.set_ylabel('Loss', size=15)
ax.legend(fontsize=15)
ax = fig.add_subplot(1, 2, 2)
ax.plot(x_arr, hist['accuracy'], '-o', label='Train acc.')
ax.plot(x_arr, hist['val_accuracy'], '--<', label='Validation acc.')
ax.legend(fontsize=15)
ax.set_xlabel('Epoch', size=15)
ax.set_ylabel('Accuracy', size=15)

#plt.savefig('figures/15_12.png', dpi=300)
plt.show()


In [ ]:
test_results = model.evaluate(mnist_test.batch(20))
print('\nTest Acc. {:.2f}%'.format(test_results[1]*100))


In [ ]:
batch_test = next(iter(mnist_test.batch(12)))

preds = model(batch_test[0])

tf.print(preds.shape)
preds = tf.argmax(preds, axis=1)
print(preds)

fig = plt.figure(figsize=(12, 4))
for i in range(12):
    ax = fig.add_subplot(2, 6, i+1)
    ax.set_xticks([]); ax.set_yticks([])
    img = batch_test[0][i, :, :, 0]
    ax.imshow(img, cmap='gray_r')
    ax.text(0.9, 0.1, '{}'.format(preds[i]), 
            size=15, color='blue',
            horizontalalignment='center',
            verticalalignment='center', 
            transform=ax.transAxes)
    
#plt.savefig('figures/15_13.png', dpi=300)
plt.show()


In [ ]:
import os

if not os.path.exists('models'):
    os.mkdir('models')


model.save('models/mnist-cnn.h5')


----

Readers may ignore the next cell.

In [ ]:
! python ../.convert_notebook_to_script.py --input ch15_part1.ipynb --output ch15_part1.py
